In [19]:
import pandas as pd
import numpy as np

# ---- paths ----
MASTER_CSV = "../data/cleaned_all_participants_right.parquet"  # your master file

# ---- only read what we need ----
usecols = [
    "participant",
    "trial",
    "stim_category",
    "t_rel_ms",
    "time_ms",

    # pupil
    "pupil_right_bc",
    "pupil_right_clean",

    # gaze events
    "category_right",

    # gaze positions
    "por_right_x_px",
    "por_right_y_px",
]

# Read
df = pd.read_parquet(MASTER_CSV, columns=usecols)

# Make it smaller + faster
df["participant"] = df["participant"].astype("int16")
df["trial"] = df["trial"].astype("category")
df["stim_category"] = df["stim_category"].astype("category")
df["t_rel_ms"] = pd.to_numeric(df["t_rel_ms"], errors="coerce").astype("float32")
df["time_ms"] = pd.to_numeric(df["time_ms"], errors="coerce").astype("float64")

df["pupil_right_bc"] = pd.to_numeric(df["pupil_right_bc"], errors="coerce").astype("float32")
df["pupil_right_clean"] = pd.to_numeric(df["pupil_right_clean"], errors="coerce").astype("float32")

df["category_right"] = df["category_right"].astype("category")

df["por_right_x_px"] = pd.to_numeric(df["por_right_x_px"], errors="coerce").astype("float32")
df["por_right_y_px"] = pd.to_numeric(df["por_right_y_px"], errors="coerce").astype("float32")


if "category_right" in df.columns:
    df["category_right"] = df["category_right"].astype("category")

print(df.shape, df.columns.tolist())

(8034369, 10) ['participant', 'trial', 'stim_category', 't_rel_ms', 'time_ms', 'pupil_right_bc', 'pupil_right_clean', 'category_right', 'por_right_x_px', 'por_right_y_px']


In [ ]:
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# Pick gaze columns
# ---------------------------------------------------------
def pick_gaze_xy(df: pd.DataFrame):
    """Pick best available gaze position columns and return numeric Series x,y."""
    # prefer gaze_right_x/y if they have variance (not all zeros)
    if "gaze_right_x" in df.columns and "gaze_right_y" in df.columns:
        x = pd.to_numeric(df["gaze_right_x"], errors="coerce")
        y = pd.to_numeric(df["gaze_right_y"], errors="coerce")
        if x.notna().any() and y.notna().any():
            # if they are basically constant (e.g., all 0), fall back to POR
            if (x.std(skipna=True) > 0) or (y.std(skipna=True) > 0):
                return x, y

    # fallback: point of regard pixels (usually the real one)
    if "por_right_x_px" in df.columns and "por_right_y_px" in df.columns:
        x = pd.to_numeric(df["por_right_x_px"], errors="coerce")
        y = pd.to_numeric(df["por_right_y_px"], errors="coerce")
        if x.notna().any() and y.notna().any():
            return x, y

    raise ValueError("No usable gaze X/Y columns found (need gaze_right_x/y or por_right_x_px/y_px).")


FIX_LABELS = np.array(["fixation"])
SAC_LABELS = np.array(["saccade"])


# ---------------------------------------------------------
# Helpers: segment counts + durations + saccade amplitudes
# ---------------------------------------------------------
def count_segments(mask: np.ndarray) -> int:
    """Number of contiguous True segments."""
    if mask.size == 0:
        return 0
    d = np.diff(np.concatenate([[0], mask.view(np.int8), [0]]))
    return int((d == 1).sum())

def mean_segment_duration_ms(mask: np.ndarray, t: np.ndarray) -> float:
    """
    Mean duration of contiguous True segments, computed from segment start/end times.
    More correct than summing dt samples and dividing by count.
    """
    if mask.size == 0 or not np.any(mask):
        return np.nan

    # segment start/end indices
    d = np.diff(np.concatenate([[0], mask.view(np.int8), [0]]))
    starts = np.where(d == 1)[0]
    ends = np.where(d == -1)[0] - 1

    durs = []
    for a, b in zip(starts, ends):
        ta, tb = t[a], t[b]
        if np.isfinite(ta) and np.isfinite(tb) and tb >= ta:
            durs.append(tb - ta)

    return float(np.mean(durs)) if durs else np.nan

def mean_saccade_amplitude_by_episode(mask: np.ndarray, gx: np.ndarray, gy: np.ndarray) -> float:
    """
    Mean saccade amplitude computed per saccade episode:
    distance between first and last sample within each saccade segment.
    """
    if mask.size == 0 or not np.any(mask):
        return np.nan

    d = np.diff(np.concatenate([[0], mask.view(np.int8), [0]]))
    starts = np.where(d == 1)[0]
    ends = np.where(d == -1)[0] - 1

    amps = []
    for a, b in zip(starts, ends):
        if np.isfinite(gx[a]) and np.isfinite(gx[b]) and np.isfinite(gy[a]) and np.isfinite(gy[b]):
            amps.append(np.hypot(gx[b] - gx[a], gy[b] - gy[a]))

    return float(np.mean(amps)) if amps else np.nan


# ---------------------------------------------------------
# Main: per-trial gaze features
# ---------------------------------------------------------
def compute_trial_gaze_features(trial_df: pd.DataFrame) -> dict:
    # --- time ---
    t = pd.to_numeric(trial_df["time_ms"], errors="coerce").to_numpy()
    if t.size == 0 or np.all(~np.isfinite(t)):
        return {"mean_fix_dur_ms": np.nan, "fix_count": 0.0, "mean_sac_amp": np.nan, "sac_count": 0.0}

    # enforce monotonic-ish time (optional safety)
    # (if your data can be unsorted, uncomment next 3 lines)
    # order = np.argsort(t)
    # t = t[order]
    # trial_df = trial_df.iloc[order]

    # --- labels ---
    if "category_right" not in trial_df.columns:
        return {"mean_fix_dur_ms": np.nan, "fix_count": 0.0, "mean_sac_amp": np.nan, "sac_count": 0.0}

    cat = trial_df["category_right"].astype(str).str.strip().str.lower().to_numpy()

    # --- gaze positions ---
    x, y = pick_gaze_xy(trial_df)
    gx = x.to_numpy()
    gy = y.to_numpy()

    # --- masks ---
    is_fix = np.isin(cat, FIX_LABELS)
    is_sac = np.isin(cat, SAC_LABELS)

    # --- counts (episodes) ---
    fix_count = count_segments(is_fix)
    sac_count = count_segments(is_sac)

    # --- fixation duration (episode-based) ---
    mean_fix_dur = mean_segment_duration_ms(is_fix, t)

    # --- saccade amplitude (episode-based) ---
    mean_sac_amp = mean_saccade_amplitude_by_episode(is_sac, gx, gy)

    return {
        "mean_fix_dur_ms": float(mean_fix_dur) if np.isfinite(mean_fix_dur) else np.nan,
        "fix_count": float(fix_count),
        "mean_sac_amp": float(mean_sac_amp) if np.isfinite(mean_sac_amp) else np.nan,
        "sac_count": float(sac_count),
    }


# ---------------------------------------------------------
# Compute gaze features per trial
# ---------------------------------------------------------
trial_keys = ["participant", "trial", "stim_category"]

gaze_feat = (
    df.groupby(trial_keys, observed=True, sort=False)
      .apply(lambda g: pd.Series(compute_trial_gaze_features(g)))
      .reset_index()
)

gaze_feat.head()

SyntaxError: incomplete input (3428699048.py, line 63)

In [17]:
# pick a trial that shows mean_sac_amp = 0
one = df[(df["participant"] == 1) & (df["trial"].astype(str) == "Trial001")].copy()

# what labels do we actually have?
print(one["category_right"].astype(str).str.strip().str.lower().value_counts().head(10))

# which gaze columns are being used and do they vary?
x, y = pick_gaze_xy(one)
print("x col head:", x.head().tolist())
print("x finite:", x.notna().mean(), "x std:", float(pd.to_numeric(x, errors="coerce").std()))
print("y finite:", y.notna().mean(), "y std:", float(pd.to_numeric(y, errors="coerce").std()))

# check variation specifically during saccades
cat = one["category_right"].astype(str).str.strip().str.lower()
is_sac = cat.eq("saccade")
print("saccade rows:", int(is_sac.sum()))
print("x std during sacc:", float(pd.to_numeric(x[is_sac], errors="coerce").std()))
print("y std during sacc:", float(pd.to_numeric(y[is_sac], errors="coerce").std()))

category_right
fixation     3520
blink         562
saccade       416
-              12
separator       4
Name: count, dtype: int64
x col head: [nan, 0.0, 0.0, 0.0, 0.0]
x finite: 0.999113867966327 x std: 0.0
y finite: 0.999113867966327 y std: 0.0
saccade rows: 416
x std during sacc: 0.0
y std during sacc: 0.0


In [22]:
def compute_trial_pupil_features(trial_df: pd.DataFrame) -> dict:
    tdf = trial_df.sort_values("t_rel_ms")
    y = pd.to_numeric(tdf["pupil_right_bc"], errors="coerce").to_numpy()
    t = pd.to_numeric(tdf["t_rel_ms"], errors="coerce").to_numpy()

    ok = np.isfinite(y) & np.isfinite(t)
    y = y[ok]
    t = t[ok]

    if len(y) < 5:
        return {
            "peak_dilation": np.nan,
            "mean_response": np.nan,
            "trial_sd": np.nan,
            "mean_velocity": np.nan,
            "peak_velocity": np.nan,
        }

    peak = float(np.nanmax(y))
    mean = float(np.nanmean(y))
    sd = float(np.nanstd(y))

    # velocity = dy/dt (ms) -> you can scale to per-second if you want
    dt = np.diff(t)
    dy = np.diff(y)
    v = np.divide(dy, dt, out=np.full_like(dy, np.nan, dtype=float), where=(dt > 0))

    return {
        "peak_dilation": peak,
        "mean_response": mean,
        "trial_sd": sd,
        "mean_velocity": float(np.nanmean(v)) if np.isfinite(v).any() else np.nan,
        "peak_velocity": float(np.nanmax(np.abs(v))) if np.isfinite(v).any() else np.nan,
    }

pupil_feat = (
    df.groupby(trial_keys, observed=True, sort=False)
      .apply(lambda g: pd.Series(compute_trial_pupil_features(g)))
      .reset_index()
)

pupil_feat.head()

/var/folders/g3/x3j647bn42j_bpmd_9vl047w0000gn/T/ipykernel_99572/3280179173.py:38: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series(compute_trial_pupil_features(g)))


,participant,trial,stim_category,peak_dilation,mean_response,trial_sd,mean_velocity,peak_velocity
0,1,Trial001,federica,0.345966,-0.240484,0.279964,0.000003,0.041379
1,1,Trial001,coucou_left,-0.003110,-0.119028,0.067382,0.000081,0.000219
2,1,Trial002,front_view,0.067546,-0.026090,0.119368,0.000077,0.017939
3,1,Trial003,dog_gaze_left,0.360910,0.119949,0.121096,-0.000371,0.035382
4,1,Trial004,dog_head_left,-0.020090,-0.070589,0.044274,0.000605,0.076421


In [23]:
# 1) peak should be >= mean most of the time (not always, but usually)
((pupil_feat["peak_dilation"] >= pupil_feat["mean_response"]).mean())

# 2) mean_velocity should be ~0 on average across trials
pupil_feat["mean_velocity"].describe()

count    1.866000e+03
mean    -9.464358e-07
std      1.966166e-03
min     -4.255034e-02
25%     -9.467029e-05
50%     -1.605530e-06
75%      5.329585e-05
max      2.510648e-02
Name: mean_velocity, dtype: float64

In [11]:
trial_keys = ["participant", "trial"]

# keep stim_category from pupil (or decide precedence)
trial_feat = (
    pupil_feat
    .merge(gaze_feat.drop(columns=["stim_category"], errors="ignore"),
           on=trial_keys, how="left")   # LEFT keeps all pupil trials
)

# labels (y)
y = trial_feat["stim_category"].astype(str)

pupil_cols = ["peak_dilation", "mean_response", "trial_sd", "mean_velocity", "peak_velocity"]
gaze_cols  = ["mean_fix_dur_ms", "fix_count", "mean_sac_amp", "sac_count"]

X_pupil_only = trial_feat[pupil_cols].copy()
X_gaze_only  = trial_feat[gaze_cols].copy()
X_combined   = trial_feat[pupil_cols + gaze_cols].copy()

print("trial_feat:", trial_feat.shape)
print("Missing gaze rows:", trial_feat["mean_fix_dur_ms"].isna().mean())
print("y classes:", y.nunique())

trial_feat: (5366, 12)
Missing gaze rows: 0.11926947446887812
y classes: 35


In [10]:
# Robust aggregation: median per participant × condition
participant_cond_feat = (
    trial_feat
    .groupby(["participant", "stim_category"], observed=True)[pupil_cols + gaze_cols]
    .median()
    .reset_index()
)

participant_cond_feat.head()

,participant,stim_category,peak_dilation,mean_response,trial_sd,mean_velocity,peak_velocity,mean_fix_dur_ms,fix_count,mean_sac_amp,sac_count
0,1,cat_eyes_left,0.935045,0.346788,0.270287,-0.001564,0.258456,163.091200,5.0,0.0,5.0
1,1,cat_head_left,0.342556,-0.662230,0.340175,0.000231,0.059317,275.266833,6.0,0.0,5.0
2,1,cat_head_right,0.609240,0.281192,0.186659,-0.000276,0.012474,NaN,0.0,NaN,0.0
3,1,cat_point_left,0.120123,-0.073915,0.090728,-0.000016,0.012335,369.601250,7.0,0.0,6.5
4,1,cat_point_right,0.049973,-0.104908,0.105296,-0.000322,0.007750,NaN,0.0,NaN,0.0


In [24]:
import pandas as pd
import numpy as np
from pathlib import Path
import pickle

# -----------------------------
# Inputs (assumed already computed)
#   pupil_feat columns: participant, trial, stim_category, peak_dilation, mean_response, trial_sd, mean_velocity, peak_velocity
#   gaze_feat  columns: participant, trial, stim_category, mean_fix_dur_ms, fix_count, mean_sac_amp, sac_count
# -----------------------------
trial_keys = ["participant", "trial", "stim_category"]

pupil_cols = ["peak_dilation", "mean_response", "trial_sd", "mean_velocity", "peak_velocity"]
gaze_cols  = ["mean_fix_dur_ms", "fix_count", "mean_sac_amp", "sac_count"]

# -----------------------------
# Merge to trial-level master features
# -----------------------------
trial_feat = (
    pupil_feat.merge(gaze_feat, on=trial_keys, how="left")  # left keeps all pupil trials; gaze may be missing
)

# Optional: drop trials missing pupil features (shouldn't happen if pupil_feat is your base)
trial_feat = trial_feat.dropna(subset=pupil_cols)

# -----------------------------
# Labels (y)
# -----------------------------
y = trial_feat["stim_category"].astype(str)

# -----------------------------
# Feature matrices (X)
# -----------------------------
X_pupil_only = trial_feat[pupil_cols].copy()
X_gaze_only  = trial_feat[gaze_cols].copy()
X_combined   = trial_feat[pupil_cols + gaze_cols].copy()

# If you prefer: fill missing gaze features with 0 (common when "no fixations" etc.)
# (Alternatively, keep NaNs and impute later in sklearn pipeline.)
X_gaze_only  = X_gaze_only.fillna(0.0)
X_combined   = X_combined.fillna(0.0)

print("trial_feat:", trial_feat.shape)
print("X_pupil_only:", X_pupil_only.shape)
print("X_gaze_only:", X_gaze_only.shape)
print("X_combined:", X_combined.shape)
print("y:", y.shape, "| classes:", y.nunique())

# -----------------------------
# Save for modelling notebook
# -----------------------------
OUTDIR = Path("../data/features")
OUTDIR.mkdir(parents=True, exist_ok=True)

# Save CSV (human-readable)
trial_feat.to_csv(OUTDIR / "trial_features_all.csv", index=False)
X_pupil_only.to_csv(OUTDIR / "X_pupil_only.csv", index=False)
X_gaze_only.to_csv(OUTDIR / "X_gaze_only.csv", index=False)
X_combined.to_csv(OUTDIR / "X_combined.csv", index=False)
y.to_frame("y").to_csv(OUTDIR / "y_labels.csv", index=False)

# Save Pickle (fast reload)
with open(OUTDIR / "features_bundle.pkl", "wb") as f:
    pickle.dump(
        {
            "trial_feat": trial_feat,
            "X_pupil_only": X_pupil_only,
            "X_gaze_only": X_gaze_only,
            "X_combined": X_combined,
            "y": y,
            "pupil_cols": pupil_cols,
            "gaze_cols": gaze_cols,
            "trial_keys": trial_keys,
        },
        f,
        protocol=pickle.HIGHEST_PROTOCOL,
    )

print("Saved to:", OUTDIR.resolve())

trial_feat: (1866, 12)
X_pupil_only: (1866, 5)
X_gaze_only: (1866, 4)
X_combined: (1866, 9)
y: (1866,) | classes: 35
Saved to: /Users/mariathurn/Desktop/CCS3/EyeTracking/ASD-pupil-project/data/features


In [25]:
import pandas as pd
from pathlib import Path

META_PATH = Path("../data/_Metadata_Participants.csv")

meta = pd.read_csv(META_PATH)

# Rename for consistency
meta = meta.rename(columns={
    "ParticipantID": "participant",
    "Class": "diagnosis",
    "CARS Score": "cars_score",
})

# Binary label: ASD = 1, TD = 0
meta["y_binary"] = meta["diagnosis"].map({"ASD": 1, "TD": 0})

# Optional: clean types
meta["participant"] = meta["participant"].astype(int)
meta["Age"] = pd.to_numeric(meta["Age"], errors="coerce")
meta["cars_score"] = pd.to_numeric(meta["cars_score"], errors="coerce")

meta.head()

,participant,Gender,Age,diagnosis,cars_score,y_binary
0,1,M,7.0,ASD,32.5,1
1,2,F,8.9,ASD,36.5,1
2,3,M,4.4,ASD,27.0,1
3,4,M,6.9,ASD,35.0,1
4,5,M,8.9,ASD,31.0,1


In [26]:
trial_feat_labeled = trial_feat.merge(
    meta[["participant", "y_binary", "diagnosis", "Age", "Gender", "cars_score"]],
    on="participant",
    how="inner"
)

print("Trials after merge:", len(trial_feat_labeled))
print(trial_feat_labeled[["participant","diagnosis","y_binary"]].drop_duplicates().head())

Trials after merge: 1866
     participant diagnosis  y_binary
0              1       ASD         1
27            10       ASD         1
38            11       ASD         1
78            13       ASD         1
134           14       ASD         1


In [27]:
pupil_cols = ["peak_dilation", "mean_response", "trial_sd", "mean_velocity", "peak_velocity"]
gaze_cols  = ["mean_fix_dur_ms", "fix_count", "mean_sac_amp", "sac_count"]

X_pupil_only = trial_feat_labeled[pupil_cols].copy()
X_gaze_only  = trial_feat_labeled[gaze_cols].copy()
X_combined   = trial_feat_labeled[pupil_cols + gaze_cols].copy()

y = trial_feat_labeled["y_binary"].astype(int)

print("X_pupil_only:", X_pupil_only.shape)
print("X_gaze_only :", X_gaze_only.shape)
print("X_combined  :", X_combined.shape)
print("y:", y.shape, "| ASD rate:", y.mean())

X_pupil_only: (1866, 5)
X_gaze_only : (1866, 4)
X_combined  : (1866, 9)
y: (1866,) | ASD rate: 0.48392282958199356


In [28]:
OUT_DIR = Path("../data/final_features")
OUT_DIR.mkdir(parents=True, exist_ok=True)

ids = trial_feat_labeled[[
    "participant", "trial", "stim_category",
    "diagnosis", "y_binary"
]]

X_pupil_only.to_csv(OUT_DIR / "X_pupil_only.csv", index=False)
X_gaze_only.to_csv(OUT_DIR / "X_gaze_only.csv", index=False)
X_combined.to_csv(OUT_DIR / "X_combined.csv", index=False)
y.to_csv(OUT_DIR / "y_binary.csv", index=False)
ids.to_csv(OUT_DIR / "trial_ids.csv", index=False)

print("Saved final feature matrices to:", OUT_DIR)

Saved final feature matrices to: ../data/final_features
